# অধ্যায় ৬: পাইপলাইন
## পাঠ ৬.২: পাইপলাইনে গ্রিড সার্চ ও ডেটা লিকেজ

আজ আমরা শিখব কীভাবে পাইপলাইনের সাথে GridSearchCV ব্যবহার করতে হয় এবং 'ডেটা লিকেজ' নামক একটি বিপদ সম্পর্কে সচেতন হতে হয়।

### A. গল্প: পরীক্ষার প্রশ্ন ফাঁস

তুমি একটি পরীক্ষা দেবে। কিন্তু পরীক্ষার আগের রাতে তুমি প্রশ্নপত্র পেয়ে গেছ! তুমি উত্তরগুলো মুখস্থ করে পরীক্ষায় ভালো করেছ। কিন্তু এটা কি প্রমাণ করে যে তুমি বিষয়টা বুঝেছ? না—কারণ পরীক্ষার উত্তর তুমি আগেই জেনে গিয়েছিলে।

ডেটা লিকেজও একই রকম। যখন ট্রেনিং ডেটা থেকে টেস্ট ডেটার কিছু তথ্য 'লিক' হয়ে যায়, তখন মডেল দেখায় ভালো করছে, কিন্তু বাস্তবে তা কাজ করে না।

### B. ডেটা লিকেজ কী?

ডেটা লিকেজ হয় যখন আমরা ট্রেনিং প্রক্রিয়ায় টেস্ট ডেটার তথ্য ব্যবহার করি। সবচেয়ে সাধারণ ভুলটি হলো:

**ভুল পদ্ধতি:** প্রথমে পুরো ডেটা স্কেল করো, তারপর ট্রেন-টেস্ট স্প্লিট করো।

এখানে সমস্যা হলো `StandardScaler.fit()` পুরো ডেটার (ট্রেন+টেস্ট) মিন ও স্ট্যান্ডার্ড ডেভিয়েশন ব্যবহার করে। ফলে টেস্ট ডেটার তথ্য ট্রেনিং ডেটায় 'লিক' হয়ে গেছে!

**সঠিক পদ্ধতি:** প্রথমে ট্রেন-টেস্ট স্প্লিট করো, তারপর শুধু ট্রেন ডেটায় `fit_transform()` এবং টেস্ট ডেটায় শুধু `transform()`।

পাইপলাইন স্বয়ংক্রিয়ভাবে এটি নিশ্চিত করে।

In [1]:
# প্রয়োজনীয় লাইব্রেরি
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import accuracy_score

# ডেটা
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
print('Breast Cancer:', X.shape)

Breast Cancer: (569, 30)


### C. ভুল পদ্ধতি: ডেটা লিকেজ (কী করবে না)

প্রথমে পুরো ডেটা স্কেল করা, তারপর স্প্লিট করা—এটা ভুল!

In [2]:
# ভুল পদ্ধতি: আগে স্কেল, পরে স্প্লিট
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)  # পুরো ডেটা ব্যবহার!

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_scaled_wrong, y, test_size=0.3, random_state=42
)

svm_w = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_w.fit(X_train_w, y_train_w)
acc_wrong = accuracy_score(y_test_w, svm_w.predict(X_test_w))
print('ভুল পদ্ধতি (Data Leakage):')
print(f'  Accuracy: {acc_wrong:.4f}')
print('  ⚠️  টেস্ট ডেটার তথ্য ফাঁস হয়ে গেছে!')

ভুল পদ্ধতি (Data Leakage):
  Accuracy: 0.9766
  ⚠️  টেস্ট ডেটার তথ্য ফাঁস হয়ে গেছে!


### D. সঠিক পদ্ধতি (কী করতে হবে)

প্রথমে স্প্লিট, তারপর শুধু ট্রেনে ফিট, টেস্টে ট্রান্সফর্ম—এটা সঠিক।

In [3]:
# সঠিক পদ্ধতি: আগে স্প্লিট, তারপর স্কেল
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler_correct = StandardScaler()
X_train_scaled = scaler_correct.fit_transform(X_train)  # শুধু ট্রেনে ফিট
X_test_scaled = scaler_correct.transform(X_test)  # টেস্টে শুধু ট্রান্সফর্ম

svm_c = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm_c.fit(X_train_scaled, y_train)
acc_correct = accuracy_score(y_test, svm_c.predict(X_test_scaled))
print('সঠিক পদ্ধতি:')
print(f'  Accuracy: {acc_correct:.4f}')
print('  ✅ ট্রেন ও টেস্ট আলাদা!')

সঠিক পদ্ধতি:
  Accuracy: 0.9766
  ✅ ট্রেন ও টেস্ট আলাদা!


### E. সঠিক পদ্ধতি ২: পাইপলাইন দিয়ে

পাইপলাইন স্বয়ংক্রিয়ভাবে সঠিক পদ্ধতি নিশ্চিত করে। আমাদের কিছু মনে রাখতে হয় না।

In [4]:
# পাইপলাইন দিয়ে (সঠিক ও সহজ)
pipe = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
)

pipe.fit(X_train_scaled, y_train_scaled) if False else pipe.fit(X_train, y_train)
# পাইপলাইন জানেই কি করতে হবে!

acc_pipe = pipe.score(X_test, y_test)
print(f'পাইপলাইন দিয়ে Accuracy: {acc_pipe:.4f}')
print('✅ পাইপলাইন লিকেজ প্রতিরোধ করে!')

পাইপলাইন দিয়ে Accuracy: 0.9766
✅ পাইপলাইন লিকেজ প্রতিরোধ করে!


### F. পাইপলাইনের সাথে GridSearchCV

পাইপলাইনের সাথে GridSearchCV ব্যবহার করলে আমরা পাইপলাইনের যেকোনো স্টেপের প্যারামিটার টিউন করতে পারি। প্যারামিটারের নাম হয় `স্টেপের নাম__প্যারামিটার` (ডাবল আন্ডারস্কোর)।

যেমন:
- `svc__C`: SVM-এর C প্যারামিটার
- `svc__gamma`: SVM-এর gamma প্যারামিটার
- `scaler__with_mean`: স্কেলারের প্যারামিটার

In [5]:
# পাইপলাইন + GridSearchCV
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42))
])

param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': [0.001, 0.01, 0.1, 1],
    'svc__kernel': ['rbf', 'linear']
}

grid = GridSearchCV(
    pipe, param_grid, cv=5,
    scoring='accuracy', n_jobs=-1, verbose=0
)

# ফিট
grid.fit(X_train, y_train)
print('সেরা প্যারামিটার:')
for param, value in grid.best_params_.items():
    print(f'  {param}: {value}')
print(f'\nসেরা CV স্কোর: {grid.best_score_:.4f}')
print(f'টেস্ট Accuracy: {grid.score(X_test, y_test):.4f}')
print('\n✅ GridSearchCV পাইপলাইনের সাথে নিরাপদে কাজ করে—লিকেজ হয় না!')

সেরা প্যারামিটার:
  svc__C: 10
  svc__gamma: 0.01
  svc__kernel: rbf

সেরা CV স্কোর: 0.9698
টেস্ট Accuracy: 0.9883

✅ GridSearchCV পাইপলাইনের সাথে নিরাপদে কাজ করে—লিকেজ হয় না!


### G. পাইপলাইন ছাড়া GridSearchCV-তে লিকেজ

যদি পাইপলাইন না ব্যবহার করি এবং ম্যানুয়ালি স্কেল করি, তাহলে GridSearchCV-র ক্রস-ভ্যালিডেশনের সময়ও লিকেজ হতে পারে। কারণ GridSearchCV প্রতিটি ফোল্ডে আলাদাভাবে স্কেলিং প্রয়োগ করে না যদি আমরা আগেই ডেটা স্কেল করে ফেলি।

**পাইপলাইন সমাধান:** পাইপলাইন নিশ্চিত করে যে প্রতিটি CV ফোল্ডে স্কেলিং শুধু ট্রেন ফোল্ডে ফিট হয়—টেস্ট ফোল্ডে নয়। এটাই সবচেয়ে নিরাপদ পদ্ধতি।

In [6]:
# আরও জটিল পাইপলাইন
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

pipe_complex = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('knn', KNeighborsClassifier())
])

param_grid_complex = {
    'pca__n_components': [5, 10, 15, 20],
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance']
}

grid_complex = GridSearchCV(
    pipe_complex, param_grid_complex, 
    cv=5, scoring='accuracy', n_jobs=-1
)
grid_complex.fit(X_train, y_train)
print('জটিল পাইপলাইন GridSearch:')
print(f'  সেরা প্যারামিটার: {grid_complex.best_params_}')
print(f'  সেরা CV স্কোর: {grid_complex.best_score_:.4f}')
print(f'  টেস্ট Accuracy: {grid_complex.score(X_test, y_test):.4f}')
print(f'\n  মোট {len(grid_complex.cv_results_["params"])} টি কম্বিনেশন চেক করা হয়েছে')

জটিল পাইপলাইন GridSearch:
  সেরা প্যারামিটার: {'knn__n_neighbors': 5, 'knn__weights': 'distance', 'pca__n_components': 10}
  সেরা CV স্কোর: 0.9622
  টেস্ট Accuracy: 0.9649

  মোট 32 টি কম্বিনেশন চেক করা হয়েছে


### H. ডেটা লিকেজের অন্যান্য উদাহরণ

১. **ফিচার সিলেকশন আগে করা:** ট্রেন-টেস্ট স্প্লিটের আগে ফিচার সিলেক্ট করলে টেস্ট ডেটার তথ্য ফাঁস হয়
২. **টার্গেট এনকোডিং:** টার্গেটের গড় ব্যবহার করে ফিচার তৈরি করলে লিকেজ হয়
৩. **সময়-সিরিজ ডেটা শাফল করা:** সময়ের ক্রম ভেঙে এলোমেলো করলে ভবিষ্যতের তথ্য অতীতে চলে আসে

**সোনালি নিয়ম:** ট্রেনিং-এর সময় টেস্ট ডেটার কোনো তথ্যই ব্যবহার করো না। পাইপলাইন এই নিয়ম মানতে সাহায্য করে।

### I. তুমি কি বুঝতে পেরেছ?

**প্রশ্ন ১:** ডেটা লিকেজ কী এবং কেন এটা খারাপ?

**প্রশ্ন ২:** পাইপলাইন কীভাবে ডেটা লিকেজ প্রতিরোধ করে?

**প্রশ্ন ৩:** GridSearchCV-তে পাইপলাইনের প্যারামিটার কিভাবে এক্সেস করা যায়?

**প্রশ্ন ৪:** ডেটা লিকেজের আরেকটি উদাহরণ দাও।

### J. সারসংক্ষেপ

আজ আমরা শিখলাম:
✅ ডেটা লিকেজ হয় যখন ট্রেনিং-এ টেস্ট ডেটার তথ্য ব্যবহার করা হয়
✅ লিকেজের কারণে মডেল দেখায় ভালো, কিন্তু বাস্তবে খারাপ কাজ করে
✅ পাইপলাইন স্বয়ংক্রিয়ভাবে লিকেজ প্রতিরোধ করে
✅ পাইপলাইনের সাথে GridSearchCV নিরাপদে কাজ করে
✅ প্যারামিটার নাম: `স্টেপের_নাম__প্যারামিটার_নাম`
✅ পাইপলাইনই GridSearchCV-র সাথে ব্যবহারের সবচেয়ে নিরাপদ পদ্ধতি

এভাবে আমরা অধ্যায় ৬ শেষ করলাম। পরবর্তী অধ্যায়ে আমরা টেক্সট ডেটা নিয়ে কাজ করব!